# 📘 CIFAR-10 Image Classification Learning Project
## Build and Compare **ANN vs CNN** on CIFAR-10

This notebook is designed for **students and beginners** to learn:
- How image classification works
- Why **CNN performs better than ANN**
- How architecture impacts performance
- How training strategies improve results

🎯 **Learning Goal:** Understand the complete DL pipeline by **reading the markdown + running the ready code**.

# 🧠 Problem Statement
Build an image classification model on the **CIFAR-10 dataset** using:

1. **Artificial Neural Network (ANN)**
2. **Convolutional Neural Network (CNN)**

Then compare:
- Accuracy
- Loss curves
- Generalization
- Training strategies (dropout, batch norm, augmentation)

---
### 📦 CIFAR-10 Classes
Airplane, Automobile, Bird, Cat, Deer, Dog, Frog, Horse, Ship, Truck

In [7]:
import tensorflow as tf
from tensorflow.keras import layers, models
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

print("TensorFlow version:", tf.__version__)

TensorFlow version: 2.20.0


In [8]:
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns

# 📥 Load Dataset
We use **CIFAR-10**, which contains **60,000 color images of size 32×32×3**.
- 50,000 training images
- 10,000 test images

In [ ]:
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.cifar10.load_data()

class_names = ['airplane','automobile','bird','cat','deer',
               'dog','frog','horse','ship','truck']

print("Train shape:", x_train.shape)
print("Test shape:", x_test.shape)

 23183360/170498071 ━━━━━━━━━━━━━━━━━━━━ 56:46 23us/step

In [ ]:
print("="*50)
print("Dataset Information")
print("="*50)

print(f"Training Images Shape : {x_train.shape}")
print(f"Training Labels Shape : {y_train.shape}")
print(f"Testing Images Shape  : {x_test.shape}")
print(f"Testing Labels Shape  : {y_test.shape}")

print("\nClass Labels:")

for i, label in enumerate(class_names):
    print(f"{i} : {label}")

## 🖼️ Visualize Sample Images

In [ ]:
plt.figure(figsize=(10,5))
for i in range(10):
    plt.subplot(2,5,i+1)
    plt.imshow(x_train[i])
    plt.title(class_names[y_train[i][0]])
    plt.axis("off")
plt.tight_layout()
plt.show()

# 🧹 Preprocessing
We normalize pixel values from **0–255 → 0–1** so training becomes stable.

In [ ]:
x_train_norm = x_train / 255.0
x_test_norm = x_test / 255.0

x_train_flat = x_train_norm.reshape(len(x_train_norm), -1)
x_test_flat = x_test_norm.reshape(len(x_test_norm), -1)

# 🔹 Part 1: ANN Model
ANN treats images as **flat vectors**, so it cannot preserve spatial features.
This helps students understand **why CNN is better for images**.

In [ ]:
ann_model = models.Sequential([
    layers.Dense(512, activation='relu', input_shape=(3072,)),
    layers.Dropout(0.3),
    layers.Dense(256, activation='relu'),
    layers.Dense(10, activation='softmax')
])

ann_model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

ann_history = ann_model.fit(
    x_train_flat, y_train,
    epochs=10,
    validation_split=0.1,
    batch_size=64
)

In [ ]:
ann_test_loss, ann_test_acc = ann_model.evaluate(x_test_flat, y_test)
print("ANN Test Accuracy:", ann_test_acc)

In [ ]:
print("="*60)
print("ANN MODEL SUMMARY")
print("="*60)

ann_model.summary()

# 🔹 Part 2: CNN Model
CNN preserves **spatial relationships** using:
- Convolution layers
- Pooling
- Feature extraction
- Hierarchical learning

This is why CNN performs much better for image tasks.

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping

early_stop = EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True
)

In [ ]:
cnn_model = models.Sequential([
    layers.Conv2D(32, (3,3), activation='relu', input_shape=(32,32,3)),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2,2)),

    layers.Conv2D(64, (3,3), activation='relu'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2,2)),

    layers.Conv2D(128, (3,3), activation='relu'),
    layers.Flatten(),

    layers.Dense(128, activation='relu'),
    layers.Dropout(0.4),

    layers.Dense(10, activation='softmax')
])

cnn_model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

cnn_history = cnn_model.fit(
    x_train_norm,
    y_train,
    epochs=20,
    batch_size=64,
    validation_split=0.2,
    callbacks=[early_stop]
)

In [ ]:
cnn_test_loss, cnn_test_acc = cnn_model.evaluate(
    x_test_norm,
    y_test,
    verbose=1
)

print(f"CNN Test Accuracy : {cnn_test_acc:.4f}")
print(f"CNN Test Loss     : {cnn_test_loss:.4f}")

In [ ]:
print("="*60)
print("CNN MODEL SUMMARY")
print("="*60)

cnn_model.summary()

## 📈 Compare Learning Curves

In [ ]:
plt.figure(figsize=(10,5))

plt.plot(
    ann_history.history["val_accuracy"],
    label="ANN Validation Accuracy",
    linewidth=2
)

plt.plot(
    cnn_history.history["val_accuracy"],
    label="CNN Validation Accuracy",
    linewidth=2
)

plt.xlabel("Epoch")
plt.ylabel("Validation Accuracy")
plt.title("ANN vs CNN Validation Accuracy")
plt.legend()
plt.grid(True)

plt.show()

# 🚀 Training Strategy Upgrade: Data Augmentation
This strategy improves generalization by generating transformed images.

In [ ]:
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1)
])

aug_cnn_model = models.Sequential([
    data_augmentation,

    layers.Conv2D(32,3,activation="relu",input_shape=(32,32,3)),
    layers.MaxPooling2D(),

    layers.Conv2D(64,3,activation="relu"),
    layers.MaxPooling2D(),

    layers.Flatten(),

    layers.Dense(128,activation="relu"),

    layers.Dropout(0.4),

    layers.Dense(10,activation="softmax")
])

aug_cnn_model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

print("Data Augmentation model created successfully.")

# 📊 Final Comparison Table

In [ ]:
comparison = pd.DataFrame({
    "Model": ["ANN", "CNN"],
    "Test Accuracy": [ann_test_acc, cnn_test_acc]
})

print(comparison)

plt.figure(figsize=(6,5))

bars = plt.bar(
    comparison["Model"],
    comparison["Test Accuracy"]
)

plt.title("Model Accuracy Comparison")
plt.ylabel("Accuracy")

for bar in bars:
    plt.text(
        bar.get_x() + bar.get_width()/2,
        bar.get_height(),
        f"{bar.get_height():.3f}",
        ha="center"
    )

plt.show()

In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns
import numpy as np

predictions = cnn_model.predict(x_test_norm)
predicted_labels = np.argmax(predictions, axis=1)
true_labels = y_test.flatten()

cm = confusion_matrix(true_labels, predicted_labels)

plt.figure(figsize=(10,8))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=class_names,
            yticklabels=class_names)
plt.title("CNN Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

In [ ]:
from sklearn.metrics import classification_report

print(classification_report(
    true_labels,
    predicted_labels,
    target_names=class_names
))

In [ ]:
plt.figure(figsize=(15,8))

for i in range(15):
    plt.subplot(3,5,i+1)
    plt.imshow(x_test[i])
    plt.title(f"P:{class_names[predicted_labels[i]]}\nA:{class_names[true_labels[i]]}")
    plt.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(6,5))

models = ["ANN", "CNN"]
accuracy = [ann_test_acc, cnn_test_acc]

bars = plt.bar(models, accuracy)

for bar in bars:
    plt.text(
        bar.get_x()+bar.get_width()/2,
        bar.get_height(),
        f"{bar.get_height():.3f}",
        ha='center'
    )

plt.title("ANN vs CNN Accuracy Comparison")
plt.ylabel("Accuracy")
plt.show()

# 🎓 Student Learning Tasks
Try these tasks after understanding the notebook:

### ✅ Beginner Tasks
1. Increase ANN layers and observe performance
2. Change CNN filters from 32→64→128
3. Increase epochs to 20
4. Add **EarlyStopping**
5. Add **data augmentation training**

# ✅ Conclusion

This project successfully implemented both Artificial Neural Networks (ANN) and Convolutional Neural Networks (CNN) for image classification using the CIFAR-10 dataset.

### Key Findings

- ANN provided a baseline model but could not effectively capture spatial information.
- CNN significantly improved classification accuracy by learning image features through convolutional layers.
- Batch Normalization and Dropout enhanced model stability and reduced overfitting.
- EarlyStopping improved the training strategy by restoring the best-performing model.
- Data Augmentation was introduced to improve generalization on unseen data.

### Conclusion

The CNN model outperformed the ANN model in terms of accuracy and overall image classification capability, demonstrating why CNNs are the preferred architecture for computer vision tasks.